In [0]:
# Databricks Notebook
# 6009 Assignment 1 - Step 1 Data Cleaning (Green Taxi Data Denoising)

# Environment Information
print("Spark Version:", spark.version)
print("Platform: Databricks")
print("="*50)

# Import packages
from pyspark.sql import functions as F

# ====================== Path to Merged Green Taxi Dataset ======================
file_path = "/Volumes/workspace/6009小组作业/nyctlcdataset/combined data/green_combined(1).parquet"

# 1. Read raw data
df = spark.read.parquet(file_path)
print("原始数据行数:", df.count())

# 2. Data denoising
df_clean = df.distinct() \
             .filter(F.col("lpep_pickup_datetime").isNotNull()) \
             .filter(F.col("lpep_dropoff_datetime").isNotNull()) \
             .filter(F.col("trip_distance") > 0) \
             .filter(F.col("fare_amount") > 0) \
             .filter(F.col("fare_amount") <= 1000) \
             .filter(F.col("passenger_count").between(1, 6)) \
             .filter(F.year("lpep_pickup_datetime") == 2025) \
             .filter(F.col("lpep_dropoff_datetime") > F.col("lpep_pickup_datetime"))

# 3. View denoising results
print("Denoised data row count:", df_clean.count())
print("Denoised data preview:")
df_clean.show(5)

# 4.Save processed dataset
output_path = "/Volumes/workspace/6009小组作业/nyctlcdataset/cleaned_data(for combined_data)/cleaned_green_combined(1)"
df_clean.write.mode("overwrite").parquet(output_path)

print("\n✅ Green taxi data denoising completed!")
print("✅ Cleaned dataset saved to:", output_path)

Spark Version: 4.1.0
Platform: Databricks
原始数据行数: 543139
去噪后数据行数: 468357
去噪后数据预览：
+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+------------------+---------+
|VendorID|lpep_pickup_datetime|lpep_dropoff_datetime|store_and_fwd_flag|RatecodeID|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|ehail_fee|improvement_surcharge|total_amount|payment_type|trip_type|congestion_surcharge|cbd_congestion_fee|taxi_type|
+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+------------------+-----

In [0]:
# Databricks Notebook
# Check Step1 Green Taxi Data Cleaning Results

from pyspark.sql import functions as F

check_path = "/Volumes/workspace/6009小组作业/nyctlcdataset/cleaned_data(for combined_data)/cleaned_green_combined(1)"
df_check = spark.read.parquet(check_path)

print("====== 🔍 Step1 Green Taxi Cleaning Check ======\n")

# 1. Check total row count
total = df_check.count()
print(f"✅ Total rows after cleaning: {total}")

# 2. Check for duplicate rows
distinct_count = df_check.distinct().count()
print(f"✅ Row count after deduplication: {distinct_count}")
print(f"✅ Fully deduplicated: {total == distinct_count}")

# 3. Check for NULL pickup time
null_pickup = df_check.filter(F.col("lpep_pickup_datetime").isNull()).count()
print(f"✅ Rows with NULL pickup time: {null_pickup}")

# 4. Check for NULL dropoff time
null_dropoff = df_check.filter(F.col("lpep_dropoff_datetime").isNull()).count()
print(f"✅ Rows with NULL dropoff time: {null_dropoff}")

# 5. Check for invalid trip distances (<= 0)
invalid_distance = df_check.filter(F.col("trip_distance") <= 0).count()
print(f"✅ Rows with trip_distance <= 0: {invalid_distance}")

# 6. Check for fare amount out of range (<=0 or >1000)
invalid_fare = df_check.filter((F.col("fare_amount") <= 0) | (F.col("fare_amount") > 1000)).count()
print(f"✅ Rows with invalid fare amount: {invalid_fare}")

# 7. Check if passenger_count is between 1 and 6
invalid_passenger = df_check.filter(~F.col("passenger_count").between(1, 6)).count()
print(f"✅ Rows with invalid passenger count: {invalid_passenger}")

# 8. Check if all data is from 2025
invalid_year = df_check.filter(F.year("lpep_pickup_datetime") != 2025).count()
print(f"✅ Rows not from 2025: {invalid_year}")

# 9. Check for dropoff time <= pickup time
invalid_time_order = df_check.filter(F.col("lpep_dropoff_datetime") <= F.col("lpep_pickup_datetime")).count()
print(f"✅ Rows with incorrect time logic: {invalid_time_order}")

print("\n====== ✅ Check Complete ======")
print("If all check items equal 0 → Data is completely correct!")

====== 🔍 开始检查 Step1 数据清洗结果 ======

✅ 清洗后总行数: 468357
✅ 去重后行数: 468357
✅ 是否完全去重: True
✅ 上车时间为空的行数: 0
✅ 下车时间为空的行数: 0
✅ 行程距离 <=0 的行数: 0
✅ 车费异常的行数: 0
✅ 乘客数异常的行数: 0
✅ 非2025年的行数: 0
✅ 时间逻辑错误的行数: 0

====== ✅ 检查完成 ======
如果所有检查项都等于 0 → 数据完全正确！


In [0]:
# Databricks Notebook
# 6009 Assignment 1 - Step 2 Time Feature Extraction (Green Taxi Time Dimension)

from pyspark.sql import functions as F

# ====================== Read cleaned green taxi dataset (read entire folder)======================
cleaned_data_path = "/Volumes/workspace/6009小组作业/nyctlcdataset/cleaned_data(for combined_data)/cleaned_green_combined(1)"
df_clean = spark.read.parquet(cleaned_data_path)

print("✅ Successfully read cleaned green taxi dataset")
print(f"Dataset row count:  {df_clean.count()}")
df_clean.printSchema()
df_clean.show(3)

# ====================== Time Dimension Extraction ======================
# Extract four time features based on lpep_pickup_datetime (green taxi specific field)
df_with_time = df_clean \
    .withColumn("Hour", F.hour("lpep_pickup_datetime")) \
    .withColumn("DayOfWeek", F.dayofweek("lpep_pickup_datetime")) \
    .withColumn("Month", F.month("lpep_pickup_datetime")) \
    .withColumn("IsWeekend", 
                F.when(F.dayofweek("lpep_pickup_datetime").isin(1, 7), True)  # Spark 中 1=周日, 7=周六
                 .otherwise(False))

print("\n✅ Time dimension extraction completed!")
print("New field preview:")
df_with_time.select("lpep_pickup_datetime", "Hour", "DayOfWeek", "Month", "IsWeekend").show(10)

# ====================== Save Processed Dataset ======================
output_path = "/Volumes/workspace/6009小组作业/nyctlcdataset/cleaned_data(for combined_data)/green_tripdata_with_time_features"
df_with_time.write.mode("overwrite").parquet(output_path)

print("\n✅ Green taxi dataset with time features saved to:")
print(output_path)

# ====================== ✅ Statistics of final exported dataset: row count + column count + all column names ======================
# Read the final saved file
df_final = spark.read.parquet(output_path)

# Row count
row_count = df_final.count()

# Column count
col_count = len(df_final.columns)

# All column names
column_names = df_final.columns

print("\n" + "="*60)
print("📊 Final Exported Dataset Statistics (Green Taxi + Time Features)")
print("="*60)
print(f"✅ Total rows:{row_count}")
print(f"✅ Total columns:{col_count}")
print(f"✅ All column names:")
for idx, col in enumerate(column_names, 1):
    print(f"   {idx}. {col}")
print("="*60)

✅ 成功读取清洗后绿色出租车数据集
数据集行数: 468357
root
 |-- VendorID: integer (nullable = true)
 |-- lpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- lpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- ehail_fee: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_type: double (nullable = true)
 |-- trip_type: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)

In [0]:
# ====================== Check Green Taxi Time Features ======================
# 1. Count orders per hour (check if Hour is normal, not all zeros)
df_with_time.groupBy("Hour").count().orderBy("Hour").show(24)

# 2. Random preview of 20 records (check time, hour, day of week are correct)
df_with_time.select(
    "lpep_pickup_datetime", 
    "Hour", 
    "DayOfWeek", 
    "Month", 
    "IsWeekend"
).orderBy(F.rand()).show(20)

# 3. Count orders for weekend vs. weekday (optional, for assignment report)
df_with_time.groupBy("IsWeekend").count().show()

+----+-----+
|Hour|count|
+----+-----+
|   0| 7631|
|   1| 5159|
|   2| 3536|
|   3| 2687|
|   4| 2289|
|   5| 2897|
|   6| 8082|
|   7|17636|
|   8|22356|
|   9|23664|
|  10|23648|
|  11|23553|
|  12|25315|
|  13|25548|
|  14|29302|
|  15|32575|
|  16|35959|
|  17|38285|
|  18|37053|
|  19|29487|
|  20|22752|
|  21|19722|
|  22|16720|
|  23|12501|
+----+-----+

+--------------------+----+---------+-----+---------+
|lpep_pickup_datetime|Hour|DayOfWeek|Month|IsWeekend|
+--------------------+----+---------+-----+---------+
| 2025-06-15 21:36:20|  21|        1|    6|     true|
| 2025-01-16 07:41:38|   7|        5|    1|    false|
| 2025-10-31 06:20:40|   6|        6|   10|    false|
| 2025-01-29 20:57:41|  20|        4|    1|    false|
| 2025-01-09 21:35:56|  21|        5|    1|    false|
| 2025-09-09 10:51:52|  10|        3|    9|    false|
| 2025-08-14 15:24:28|  15|        5|    8|    false|
| 2025-04-28 10:23:27|  10|        2|    4|    false|
| 2025-05-23 21:40:23|  21|        6|    